# Multimodal Exercise Classification & Modality Ablation


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix

def compute_multiclass_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specs = []
    for i in range(n_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return np.mean(specs)

master_df = pd.read_csv(os.path.join('data', 'master_dataset_index.csv'))
tr_feats = pd.read_csv(os.path.join('data', 'train_adv_advanced_features.csv'))
te_feats = pd.read_csv(os.path.join('data', 'test_adv_advanced_features.csv'))
df_feats = pd.concat([tr_feats, te_feats], ignore_index=True)

target_conds = ['Deadlift', 'Deep Squat', 'Forward Lunge', 'Stand-sit transition']
df_filt = master_df[master_df['condition_name'].isin(target_conds)].copy()
df_feats['subject_id'] = df_filt['subject_id'].values
df_feats['label'] = df_filt['condition_name'].values

le = LabelEncoder()
y = le.fit_transform(df_feats['label'])
groups = df_feats['subject_id'].values
n_classes = len(le.classes_)

semg_cols = ['semg_rms', 'semg_std', 'semg_p25', 'semg_p75', 'semg_p95']
amg_cols = ['amg_rms', 'amg_std', 'amg_p95']
kin_cols = ['angle_mean', 'angle_std', 'angle_range', 'angle_p25', 'angle_p75', 'angle_p95']

modalities = {
    'sEMG-only (Single)': semg_cols,
    'AMG-only (Single)': amg_cols,
    'Kinematics-only (Single)': kin_cols,
    'sEMG + AMG (Dual)': semg_cols + amg_cols,
    'sEMG + Kinematics (Dual)': semg_cols + kin_cols,
    'AMG + Kinematics (Dual)': amg_cols + kin_cols,
    'Combined Multimodal (Triple)': semg_cols + amg_cols + kin_cols
}

results = []
gkf = GroupKFold(n_splits=5)

for mod_name, cols in modalities.items():
    X = df_feats[cols].values
    accs, f1s, sens_list, specs_list, aucs = [], [], [], [], []

    for tr_idx, val_idx in gkf.split(X, y, groups):
        X_tr, X_va = X[tr_idx], X[val_idx]
        y_tr, y_va = y[tr_idx], y[val_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_va_s = scaler.transform(X_va)
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
        clf.fit(X_tr_s, y_tr)
        y_pred = clf.predict(X_va_s)
        y_prob = clf.predict_proba(X_va_s)
        accs.append(accuracy_score(y_va, y_pred))
        f1s.append(f1_score(y_va, y_pred, average='macro'))
        sens_list.append(recall_score(y_va, y_pred, average='macro'))
        specs_list.append(compute_multiclass_specificity(y_va, y_pred, n_classes))
        try:
            aucs.append(roc_auc_score(y_va, y_prob, multi_class='ovr', average='macro'))
        except Exception:
            pass

    results.append({
        'Modality': mod_name,
        'Features (D)': len(cols),
        'Accuracy (%)': f'{np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%',
        'F1-Macro': f'{np.mean(f1s):.4f}',
        'Sensitivity (%)': f'{np.mean(sens_list)*100:.2f}%',
        'Specificity (%)': f'{np.mean(specs_list)*100:.2f}%',
        'ROC-AUC': f'{np.nanmean(aucs):.4f}'
    })

df_res = pd.DataFrame(results)
display(df_res)
df_res.to_csv(os.path.join('data', 'exp1_ablation_results.csv'), index=False)

